# MOSS-Image on Google Cloud TPU (`torch_tpu` / TorchTPU)

This notebook targets **Google Colab `TPU v6e-1` (32 GiB HBM)** using **OpenMOSS-Team/MOSS-Image-Base** and Google's native PyTorch backend (`torch_tpu`).

**Before running:** Select `Runtime -> Change runtime type -> v6e-1 TPU` (requires Python 3.11+).

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/MOSS-Image")
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", "feat/torchtpu-support",
                    "https://github.com/azure-swing/MOSS-Image.git", str(repo)], check=True)
subprocess.run(["git", "checkout", "feat/torchtpu-support"], cwd=repo, check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "feat/torchtpu-support"], cwd=repo, check=True)
subprocess.run(["bash", "scripts/setup-tpu.sh"], cwd=repo, check=True)

cache = Path("/content/tpu_cache/tier3")
cache.mkdir(parents=True, exist_ok=True)
os.environ["TORCH_TPU_TIER3_COMPILATION_CACHE_ROOT"] = str(cache)

def run_tpu(*args):
    subprocess.run(["python3", "scripts/run_tpu.py", *args], cwd=repo, check=True)

## 1. Probe TPU & `torch_tpu` Runtime
Verify `device="tpu"`, HBM capacity, and BF16 MXU matmul execution.

In [ ]:
run_tpu("--mode", "probe")

## 2. Generate Image (Staged Qwen3.5-9B + Static-Bucket Compiled 5B DiT)
- **Stage 1 (`text_encoder`)** runs unpadded individual prompt forwards (preserving Qwen3.5 GDN numerics) and caches embeddings on CPU.
- **Stage 2 (`transformer` + `vae`)** pads embeddings to fixed static buckets (`512/1024/2048/4096`) with SDPA attention masking and compiles with `torch.compile(..., backend="tpu", dynamic=False, fullgraph=True)`.

In [ ]:
from IPython.display import Image, display

output = "outputs/colab_tpu_sample.png"
run_tpu("--mode", "generate", "--offload", "model", "--compile", "--steps", "28",
        "--seed", "42", "--output", output,
        "--prompt", "An editorial poster reading 'TORCHTPU X MOSS-IMAGE 2026' above a ceramic teapot.")
display(Image(filename=str(repo / output)))

## 3. Benchmark: Strict Eager vs `DEFER_AND_FUSE` vs `torch.compile(backend="tpu")`

In [ ]:
run_tpu("--mode", "benchmark", "--bench-modes", "eager,defer_and_fuse,compile",
        "--height", "1024", "--width", "1024", "--steps", "28", "--runs", "2",
        "--save-report", "outputs/tpu_benchmark.json")

## 4. Optional: Decoupled text encoding and generation

Encode on a host with enough memory for the BF16 text encoder. The saved embeddings can be copied to a separate generation host.

In [ ]:
# run_tpu("--mode", "encode-only", "--prompt", "A neon rainy Tokyo alley",
#         "--save-embeds", "outputs/embeds.pt")
# run_tpu("--mode", "generate", "--prompt-embeds", "outputs/embeds.pt",
#         "--offload", "none", "--compile", "--output", "outputs/decoupled.png")